In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from typing import List, Tuple, Optional
import time

In [ ]:
def visualize_board(board: List[int], n: int, title: str = "N-Queens Solution"):
    fig, ax = plt.subplots(figsize=(8, 8))
    for i in range(n):
        for j in range(n):
            color = 'white' if (i + j) % 2 == 0 else 'lightgray'
            ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor=color, edgecolor='black'))
    for row, col in enumerate(board):
        if col != -1:
            ax.text(col + 0.5, row + 0.5, '♛', fontsize=40, ha='center', va='center')
    ax.set_xlim(0, n)
    ax.set_ylim(0, n)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')
    ax.set_title(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
example = [1, 3, 0, 2]  
visualize_board(example, 4, "Example: 4-Queens Solution")

In [ ]:
def is_safe(board: List[int], row: int, col: int) -> bool:
    for i in range(row):
        if board[i] == col:
            return False
        if abs(board[i] - col) == abs(i - row):
            return False
    return True
def count_conflicts(board: List[int], n: int) -> int:
    conflicts = 0
    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:
                conflicts += 1
            elif abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1
    return conflicts
def is_solution(board: List[int], n: int) -> bool:
    return count_conflicts(board, n) == 0

In [ ]:
def solve_backtracking(n: int) -> Tuple[Optional[List[int]], int]:
    board = [-1] * n
    nodes_explored = [0]
    def backtrack(row: int) -> bool:
        nodes_explored[0] += 1
        if row == n:
            return True
        for col in range(n):
            if is_safe(board, row, col):
                board[row] = col
                if backtrack(row + 1):
                    return True
                board[row] = -1
        return False
    start_time = time.time()
    success = backtrack(0)
    end_time = time.time()
    print(f"\nBacktracking Results (N={n}):")
    print(f"Solution found: {success}")
    print(f"Nodes explored: {nodes_explored[0]}")
    print(f"Time taken: {end_time - start_time:.4f} seconds")
    return board if success else None, nodes_explored[0]
n = 8
solution, nodes = solve_backtracking(n)
if solution:
    print(f"Solution: {solution}")
    visualize_board(solution, n, f"Backtracking Solution ({n}-Queens)")

In [ ]:
def solve_hill_climbing(n: int, max_iterations: int = 1000) -> Tuple[List[int], int, List[int]]:
    board = [random.randint(0, n-1) for _ in range(n)]
    iterations = 0
    conflict_history = []
    start_time = time.time()
    while iterations < max_iterations:
        current_conflicts = count_conflicts(board, n)
        conflict_history.append(current_conflicts)
        if current_conflicts == 0:
            break
        best_board = board[:]
        best_conflicts = current_conflicts
        for row in range(n):
            original_col = board[row]
            for col in range(n):
                if col != original_col:
                    board[row] = col
                    conflicts = count_conflicts(board, n)
                    if conflicts < best_conflicts:
                        best_conflicts = conflicts
                        best_board = board[:]
            board[row] = original_col
        if best_conflicts >= current_conflicts:
            break
        board = best_board
        iterations += 1
    end_time = time.time()
    final_conflicts = count_conflicts(board, n)
    success = final_conflicts == 0
    print(f"\nHill Climbing Results (N={n}):")
    print(f"Solution found: {success}")
    print(f"Final conflicts: {final_conflicts}")
    print(f"Iterations: {iterations}")
    print(f"Time taken: {end_time - start_time:.4f} seconds")
    return board, iterations, conflict_history
n = 8
solution, iters, history = solve_hill_climbing(n)
print(f"Solution: {solution}")
if is_solution(solution, n):
    visualize_board(solution, n, f"Hill Climbing Solution ({n}-Queens)")
else:
    print("\nGot stuck in local optimum. Try running again or use different method.")
    visualize_board(solution, n, f"Hill Climbing - Local Optimum ({count_conflicts(solution, n)} conflicts)")
plt.figure(figsize=(10, 6))
plt.plot(history, linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Number of Conflicts')
plt.title('Hill Climbing Progress')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def solve_hill_climbing_restarts(n: int, max_restarts: int = 100) -> Tuple[Optional[List[int]], int, int]:
    total_iterations = 0
    start_time = time.time()
    for restart in range(max_restarts):
        board, iters, _ = solve_hill_climbing(n, max_iterations=100)
        total_iterations += iters
        if is_solution(board, n):
            end_time = time.time()
            print(f"\nHill Climbing with Restarts Results (N={n}):")
            print(f"Solution found: True")
            print(f"Total iterations: {total_iterations}")
            print(f"Number of restarts: {restart + 1}")
            print(f"Time taken: {end_time - start_time:.4f} seconds")
            return board, total_iterations, restart + 1
    end_time = time.time()
    print(f"\nHill Climbing with Restarts Results (N={n}):")
    print(f"Solution found: False (after {max_restarts} restarts)")
    print(f"Time taken: {end_time - start_time:.4f} seconds")
    return None, total_iterations, max_restarts
n = 8
print("Running Hill Climbing with Random Restarts...\n")
solution, total_iters, restarts = solve_hill_climbing_restarts(n)
if solution:
    print(f"\nSolution: {solution}")
    visualize_board(solution, n, f"Hill Climbing with Restarts ({n}-Queens)")

In [ ]:
def fitness(board: List[int], n: int) -> int:
    max_conflicts = n * (n - 1) // 2
    return max_conflicts - count_conflicts(board, n)
def crossover(parent1: List[int], parent2: List[int]) -> Tuple[List[int], List[int]]:
    n = len(parent1)
    point = random.randint(1, n - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2
def mutate(board: List[int], mutation_rate: float = 0.1) -> List[int]:
    board = board[:]
    n = len(board)
    for i in range(n):
        if random.random() < mutation_rate:
            board[i] = random.randint(0, n - 1)
    return board
def solve_genetic(n: int, population_size: int = 100, generations: int = 1000) -> Tuple[Optional[List[int]], int, List[int]]:
    population = [[random.randint(0, n-1) for _ in range(n)] for _ in range(population_size)]
    best_fitness_history = []
    max_fitness = n * (n - 1) // 2
    start_time = time.time()
    for generation in range(generations):
        fitness_scores = [(individual, fitness(individual, n)) for individual in population]
        fitness_scores.sort(key=lambda x: x[1], reverse=True)
        best_fitness_history.append(fitness_scores[0][1])
        if fitness_scores[0][1] == max_fitness:
            end_time = time.time()
            print(f"\nGenetic Algorithm Results (N={n}):")
            print(f"Solution found: True")
            print(f"Generation: {generation}")
            print(f"Time taken: {end_time - start_time:.4f} seconds")
            return fitness_scores[0][0], generation, best_fitness_history
        population = [individual for individual, _ in fitness_scores[:population_size // 2]]
        new_population = population[:]
        while len(new_population) < population_size:
            parent1 = random.choice(population)
            parent2 = random.choice(population)
            child1, child2 = crossover(parent1, parent2)
            new_population.append(mutate(child1))
            if len(new_population) < population_size:
                new_population.append(mutate(child2))
        population = new_population
    end_time = time.time()
    print(f"\nGenetic Algorithm Results (N={n}):")
    print(f"Solution found: False (after {generations} generations)")
    print(f"Best fitness achieved: {best_fitness_history[-1]}/{max_fitness}")
    print(f"Time taken: {end_time - start_time:.4f} seconds")
    return None, generations, best_fitness_history
n = 8
solution, gen, fitness_history = solve_genetic(n, population_size=100, generations=1000)
if solution:
    print(f"\nSolution: {solution}")
    visualize_board(solution, n, f"Genetic Algorithm Solution ({n}-Queens)")
plt.figure(figsize=(10, 6))
plt.plot(fitness_history, linewidth=2)
plt.xlabel('Generation')
plt.ylabel('Best Fitness')
plt.title('Genetic Algorithm Evolution')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def compare_algorithms(n: int):
    print("="*70)
    print(f"COMPARING ALGORITHMS FOR {n}-QUEENS PROBLEM")
    print("="*70)
    results = {}
    print("\n1. Testing Backtracking...")
    start = time.time()
    bt_solution, bt_nodes = solve_backtracking(n)
    bt_time = time.time() - start
    results['Backtracking'] = {
        'Success': bt_solution is not None,
        'Time': bt_time,
        'Metric': bt_nodes
    }
    print("\n2. Testing Hill Climbing with Restarts...")
    start = time.time()
    hc_solution, hc_iters, hc_restarts = solve_hill_climbing_restarts(n, max_restarts=50)
    hc_time = time.time() - start
    results['Hill Climbing'] = {
        'Success': hc_solution is not None,
        'Time': hc_time,
        'Metric': hc_restarts
    }
    print("\n3. Testing Genetic Algorithm...")
    start = time.time()
    ga_solution, ga_gen, _ = solve_genetic(n, population_size=100, generations=500)
    ga_time = time.time() - start
    results['Genetic Algorithm'] = {
        'Success': ga_solution is not None,
        'Time': ga_time,
        'Metric': ga_gen
    }
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    for algo, data in results.items():
        print(f"\n{algo}:")
        print(f"  Success: {data['Success']}")
        print(f"  Time: {data['Time']:.4f} seconds")
    return results
results = compare_algorithms(8)

In [ ]:
algorithms = list(results.keys())
times = [results[algo]['Time'] for algo in algorithms]
success = [results[algo]['Success'] for algo in algorithms]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['green' if s else 'red' for s in success]
ax1.bar(algorithms, times, color=colors, alpha=0.7)
ax1.set_ylabel('Time (seconds)')
ax1.set_title('Execution Time Comparison')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)
success_count = [1 if s else 0 for s in success]
ax2.bar(algorithms, success_count, color=['green' if s else 'red' for s in success], alpha=0.7)
ax2.set_ylabel('Success (1 = Yes, 0 = No)')
ax2.set_ylim(0, 1.2)
ax2.set_title('Success Rate')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
test_sizes = [4, 6, 8, 10, 12]
print("Testing algorithms on different board sizes...\n")
size_results = {}
for n in test_sizes:
    print(f"\nTesting N={n}...")
    if n <= 10:
        start = time.time()
        solution, nodes = solve_backtracking(n)
        bt_time = time.time() - start
    else:
        bt_time = None
    start = time.time()
    solution, iters, restarts = solve_hill_climbing_restarts(n, max_restarts=30)
    hc_time = time.time() - start
    size_results[n] = {
        'Backtracking': bt_time,
        'Hill Climbing': hc_time
    }
plt.figure(figsize=(10, 6))
sizes_bt = [n for n in test_sizes if size_results[n]['Backtracking'] is not None]
times_bt = [size_results[n]['Backtracking'] for n in sizes_bt]
times_hc = [size_results[n]['Hill Climbing'] for n in test_sizes]
plt.plot(sizes_bt, times_bt, 'o-', label='Backtracking', linewidth=2, markersize=8)
plt.plot(test_sizes, times_hc, 's-', label='Hill Climbing w/ Restarts', linewidth=2, markersize=8)
plt.xlabel('Board Size (N)')
plt.ylabel('Time (seconds)')
plt.title('Algorithm Scalability')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()